In [10]:
import os
import json
from datetime import datetime

import numpy as np

In [11]:
# long_prompts = []
# long_full = []
# for batch_id in range(4):
#     all_data = []
#     with open(f'rollout_len_b{batch_id}.jsonl', 'r') as f:
#         for line in f:
#             data = json.loads(line)
#             all_data.append(data)
#     num_tokens = np.array([data['output']['meta_info']['completion_tokens'] for data in all_data])
#     print(f'batch {batch_id}')
#     long_seq_idx = np.where(num_tokens > 8000)[0]
#     print(long_seq_idx)
#     for idx in long_seq_idx:
#         print(f'idx: {idx}')
#         print(all_data[idx]['output']['meta_info'])
#         long_prompts.append(all_data[idx]['input'])
#         long_full.append(all_data[idx])
#         print(all_data[idx]['input'])
#         print(datetime.fromtimestamp(all_data[idx]['output']['meta_info']['response_sent_to_client_ts']))
#         print(datetime.fromtimestamp(all_data[idx]['output']['meta_info']['response_sent_to_client_ts'] - all_data[idx]['output']['meta_info']['e2e_latency']))
#         print('-'*100)

all_data = []
for batch_id in range(4):
    with open(f'rollout_len_b{batch_id}.jsonl', 'r') as f:
        for line in f:
            data = json.loads(line)
            all_data.append(data)
num_tokens = np.array([data['output']['meta_info']['completion_tokens'] for data in all_data])


In [12]:
# 2 x tp2, 1 x tp4
os.makedirs("/eric-verl/ff/temp/verl/exp_script/dist/2tp2_1tp4/in", exist_ok=True)
os.makedirs("/eric-verl/ff/temp/verl/exp_script/dist/2tp2_1tp4/out", exist_ok=True)

for i in [2,3,4]:
    selected = np.where(num_tokens > i*1000)[0]
    b0 = []
    b1 = []
    b2 = []
    for idx, data in enumerate(all_data):
        if idx in selected:
            b0.append(data)
        else:
            b1.append(data)
    split_idx = int(len(b1) * 0.5)
    b2 = b1[:split_idx]
    b1 = b1[split_idx:]
    print(len(b0), len(b1), len(b2))
    with open(f'/eric-verl/ff/temp/verl/exp_script/dist/2tp2_1tp4/in/{i}k_b0.json', 'w') as f:
        json.dump(b0, f)
    with open(f'/eric-verl/ff/temp/verl/exp_script/dist/2tp2_1tp4/in/{i}k_b1.json', 'w') as f:
        json.dump(b1, f)
    with open(f'/eric-verl/ff/temp/verl/exp_script/dist/2tp2_1tp4/in/{i}k_b2.json', 'w') as f:
        json.dump(b2, f)
        
    with open(f'/eric-verl/ff/temp/verl/exp_script/dist/2tp2_1tp4/in/{i}k_b0.json', 'r') as f:
        data = json.load(f)
        print(len(data))
    with open(f'/eric-verl/ff/temp/verl/exp_script/dist/2tp2_1tp4/in/{i}k_b1.json', 'r') as f:
        data = json.load(f)
        print(len(data))
    with open(f'/eric-verl/ff/temp/verl/exp_script/dist/2tp2_1tp4/in/{i}k_b2.json', 'r') as f:
        data = json.load(f)
        print(len(data))

206 409 409
206
409
409
101 462 461
101
462
461
68 478 478
68
478
478


In [13]:
# 8 x tp1, 1 x tp4
os.makedirs("/eric-verl/ff/temp/verl/exp_script/dist/8tp1/in", exist_ok=True)
os.makedirs("/eric-verl/ff/temp/verl/exp_script/dist/8tp1/out", exist_ok=True)

total_len = len(all_data)
for i in range(0,8):
    b = all_data[i*total_len//8:(i+1)*total_len//8]
    with open(f'/eric-verl/ff/temp/verl/exp_script/dist/8tp1/in/{i}.json', 'w') as f:
        json.dump(b, f)
    with open(f'/eric-verl/ff/temp/verl/exp_script/dist/8tp1/in/{i}.json', 'r') as f:
        data = json.load(f)
        print(len(data))


128
128
128
128
128
128
128
128


In [33]:
with open('long_prompts.json', 'w') as f:
    json.dump(long_prompts, f)

In [41]:
all_data = []
with open(f'rollout_long.jsonl', 'r') as f:
    for line in f:
        data = json.loads(line)
        all_data.append(data)


In [42]:
for idx, long_prompt in enumerate(all_data):
    print(long_prompt['input'])
    print("perf for mixed batch")
    print(long_prompt['output']['meta_info']['prompt_tokens'])
    print(long_prompt['output']['meta_info']['e2e_latency'])
    print(datetime.fromtimestamp(long_prompt['output']['meta_info']['response_sent_to_client_ts'] - long_prompt['output']['meta_info']['e2e_latency']))
    print(datetime.fromtimestamp(long_prompt['output']['meta_info']['response_sent_to_client_ts']))
    print("perf for long batch")
    print(long_full[idx]['output']['meta_info']['prompt_tokens'])
    print(long_full[idx]['output']['meta_info']['e2e_latency'])
    print(datetime.fromtimestamp(long_full[idx]['output']['meta_info']['response_sent_to_client_ts'] - long_full[idx]['output']['meta_info']['e2e_latency']))
    print(datetime.fromtimestamp(long_full[idx]['output']['meta_info']['response_sent_to_client_ts']))
    print('-'*100)


<|im_start|>user
A cleaning company produces two sanitizer sprays. One spray kills 50% of germs, and another spray kills 25% of germs. However, 5% of the germs they kill are the same ones. What percentage of germs would be left after using both sanitizer sprays together? Let's think step by step and output the final answer after "####".<|im_end|>
<|im_start|>assistant

perf for mixed batch
87
206.42802929878235
2025-12-05 05:00:31.373560
2025-12-05 05:03:57.801589
perf for long batch
87
211.58700394630432
2025-12-05 02:55:08.625690
2025-12-05 02:58:40.212694
----------------------------------------------------------------------------------------------------
<|im_start|>user
Mark has two pets, a hare that runs 10 feet/second and a turtle that crawls 1 foot/second. If they're going to run a 20 foot-race, how much of a head start (in seconds) does the turtle need to finish in a tie? Let's think step by step and output the final answer after "####".<|im_end|>
<|im_start|>assistant

perf fo